# Mechanical Material Card (PMDCo / IAO): from data entry to RDF

This notebook shows how to assemble a mechanical material card and convert
it into a standardised, machine-readable RDF graph.

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

A mechanical material card in this schema captures:

- a reference to the **material** it describes (by IRI)
- **elastic constants**: density, Young's modulus, Poisson's ratio
- **mechanical properties** from a tensile test: yield strength, tensile strength,
  elongation (using TTO terms)
- a **constitutive model** with fitted parameter values and a link to the
  calibration step that produced them

Unlike process schemas (manufacturing, characterisation, simulation), a material
card is a **data artefact** (`iao:IAO_0000100`), not a process. It is the
*output* of a calibration step and the *input* to an FEM simulation.


## What the notebook does

```
example.input.json
  |  material IRI, elastic constants, properties, and constitutive model
  |
  v
Transform
  |  converts your plain JSON into a structured OO-LD document
  |
  v
RDF graph
  |  machine-readable, ontology-aligned triples
  |
  v
SHACL validation  ->  confirms the graph is structurally correct
SPARQL inspect    ->  shows the key properties and parameter values
```


## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # material-card/mechanical/PMDCo/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))


Schema : material-card/mechanical/PMDCo


## Step 1: Describe your material card

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `card_name` | yes | A name for this material card |
| `material_iri` | yes | IRI of the material this card describes |
| `description` | no | Free-text explanation |
| `density` | no | Density value and unit (e.g. `{"value": 7.93, "unit": "g/cm³"}`) |
| `youngs_modulus` | no | Young's modulus value and unit (e.g. `{"value": 193.0, "unit": "GPa"}`) |
| `poissons_ratio` | no | Poisson's ratio value (dimensionless) |
| `mechanical_properties` | no | List of TTO property records (see table below) |
| `constitutive_model` | no | Fitted flow curve model with parameter values |
| `card_id` | no | Custom IRI slug; auto-derived from `card_name` if omitted |

**Supported mechanical property types:**

| `type` value | Ontology term | Unit |
|---|---|---|
| `YieldStrength` | TTO | MPa |
| `TensileStrength` | TTO | MPa |
| `PercentageElongationAfterFracture` | TTO | % |
| `PercentageReductionOfArea` | TTO | % |
| `ProofStrength` | TTO | MPa |

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "card_name": "316L stainless steel, Hockett-Sherby v1",
  "description": "Mechanical material card for 316L austenitic stainless steel, batch 1. Includes elastic constants, TTO mechanical properties from tensile test, and Hockett-Sherby flow curve parameters calibrated from the same test.",
  "material_iri": "https://example.org/materials/316L-batch-1",
  "density": {
    "value": 7.93,
    "unit": "g/cm\u00b3"
  },
  "youngs_modulus": {
    "value": 193.0,
    "unit": "GPa"
  },
  "poissons_ratio": {
    "value": 0.29
  },
  "mechanical_properties": [
    {
      "type": "YieldStrength",
      "value": 285.0,
      "unit": "MPa"
    },
    {
      "type": "TensileStrength",
      "value": 620.0,
      "unit": "MPa"
    },
    {
      "type": "PercentageElongationAfterFracture",
      "value": 50.0,
      "unit": "%"
    }
  ],
  "constitutive_model": {
    "model_type": "Hockett-Sherby",
    "calibration_iri": "https://example.org/simulations/calibration-316L-batch-1",
    "parame

## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document.
The transform maps short property names (like `YieldStrength`) to their
full TTO ontology IRIs, and simplified unit strings (like `MPa`) to
QUDT unit IRIs.

You can also run the transform from the command line:
```
python -m jsonata -e ../specs/transform.simplified.jsonata -i example.input.json
```

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/material-card/mechanical/PMDCo/#v1.0.0",
  "type": "iao:0000100",
  "id": "material-card-316l-stainless-steel-hockett-sherby-v1",
  "label": "316L stainless steel, Hockett-Sherby v1",
  "material_iri": "https://example.org/materials/316L-batch-1",
  "description": "Mechanical material card for 316L austenitic stainless steel, batch 1. Includes elastic constants, TTO mechanical properties from tensile test, and Hockett-Sherby flow curve parameters calibrated from the same test.",
  "density": {
    "id": "material-card-316l-stainless-steel-hockett-sherby-v1-density",
    "scalar_value": 7.93,
    "scalar_unit": "g/cm\u00b3"
  },
  "youngs_modulus": {
    "id": "material-card-316l-stainless-steel-hockett-sherby-v1-youngs-modulus",
    "scalar_value": 193.0,
    "scalar_unit": "GPa"
  },
  "poissons_ratio": {
    "id": "material-card-316l-stainless-steel-hockett-sherby-v1-poissons-ratio",
    "sca

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context from
`specs/schema.oold.yaml`.

The material card node is typed as `iao:IAO_0000100` (DataSet), which is the
standard Information Artifact Ontology class for data artefacts.

In [5]:
flat = schema.to_graph(simplified)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 42 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix tto: <https://w3id.org/pmd/tto/> .
@prefix uqudt: <https://qudt.org/vocab/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/material-card-316l-stainless-steel-hockett-sherby-v1> a obo:IAO_0000100 ;
    rdfs:label "316L stainless steel, Hockett-Sherby v1" ;
    obo:OBI_0000299 <https://w3id.org/pmd/co/test/material-card-316l-stainless-steel-hockett-sherby-v1-prop-1>,
        <https://w3id.org/pmd/co/test/material-card-316l-stainless-steel-hockett-sherby-v1-prop-2>,
        <https://w3id.org/pmd/co/test/material-card-316l-stainless-steel-hockett-sherby-v1-prop-3> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/material-card

In [6]:
# Optional: save to file
out_ttl = HERE / "output_material_card.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_material_card.ttl


## Step 4: Validate against the SHACL shape

SHACL (Shapes Constraint Language) defines rules that a valid RDF graph must
satisfy. The shape file `specs/shape.ttl` checks that:

- The material card has exactly one label.
- The `material_iri` reference is an IRI (not a plain string).
- Every scalar property node has a numeric value.
- Every mechanical property node has a numeric value and a unit.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 5: Inspect the graph

The cells below use SPARQL to confirm that the material card was stored
correctly, including elastic constants, mechanical properties, and
constitutive model parameters.

You do not need to understand SPARQL to use this notebook. Just run the cells
and check that the output matches what you entered.

In [8]:
IAO     = rdflib.Namespace("http://purl.obolibrary.org/obo/IAO_")
DCTERMS = rdflib.Namespace("http://purl.org/dc/terms/")

# Print card label and material reference
card_iri  = next(flat.subjects(rdflib.RDF.type, IAO["0000100"]))
label     = flat.value(card_iri, rdflib.RDFS.label)
mat_ref   = flat.value(card_iri, DCTERMS.references)
print(f"Material card : {card_iri}")
print(f"Label         : {label}")
print(f"Material IRI  : {mat_ref}")

Material card : https://w3id.org/pmd/co/test/material-card-316l-stainless-steel-hockett-sherby-v1
Label         : 316L stainless steel, Hockett-Sherby v1
Material IRI  : https://example.org/materials/316L-batch-1


In [9]:
SPARQL_ELASTIC = """
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX pmd:   <https://w3id.org/pmd/co/>
PREFIX qudt:  <http://qudt.org/schema/qudt/>

SELECT ?property ?value ?unit
WHERE {
  ?card a iao:0000100 .
  VALUES (?prop ?property) {
    (pmd:PMD_0000025 "density")
    (pmd:PMD_0000039 "youngs_modulus")
    (pmd:PMD_0000040 "poissons_ratio")
  }
  ?card ?prop ?node .
  ?node qudt:value ?value .
  OPTIONAL { ?node qudt:hasUnit ?unit . }
}
ORDER BY ?property
"""

rows = list(flat.query(SPARQL_ELASTIC))
if rows:
    print("Elastic constants:")
    print(f"  {'Property':<22}  {'Value':>10}  Unit")
    print("  " + "-" * 45)
    for r in rows:
        val  = f"{float(r.value):>10.4g}"
        unit = str(r.unit) if r.unit else ""
        print(f"  {str(r.property):<22}  {val}  {unit}")
else:
    print("No elastic constants recorded.")

Elastic constants:
  Property                     Value  Unit
  ---------------------------------------------
  density                       7.93  g/cm³
  poissons_ratio                0.29  
  youngs_modulus                 193  GPa


In [10]:
SPARQL_MECH = """
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX tto:   <https://w3id.org/pmd/tto/>

SELECT ?type ?value ?unit
WHERE {
  ?card a iao:0000100 .
  ?card obi:0000299 ?prop .
  ?prop a ?type_iri ;
        obi:0001937 ?value .
  OPTIONAL { ?prop iao:0000039 ?unit . }
  FILTER(STRSTARTS(STR(?type_iri), STR(tto:)))
  BIND(REPLACE(STR(?type_iri), STR(tto:), "") AS ?type)
}
ORDER BY ?type
"""

rows = list(flat.query(SPARQL_MECH))
if rows:
    print("Mechanical properties (TTO):")
    print(f"  {'Property type':<40}  {'Value':>10}  Unit")
    print("  " + "-" * 60)
    for r in rows:
        val  = f"{float(r.value):>10.2f}"
        unit = str(r.unit).rsplit("/", 1)[-1] if r.unit else ""
        print(f"  {str(r.type):<40}  {val}  {unit}")
else:
    print("No mechanical properties recorded.")

Mechanical properties (TTO):
  Property type                                  Value  Unit
  ------------------------------------------------------------
  PercentageElongationAfterFracture              50.00  PERCENT
  TensileStrength                               620.00  MegaPascal
  YieldStrength                                 285.00  MegaPascal


In [11]:
SPARQL_MODEL = """
PREFIX iao:     <http://purl.obolibrary.org/obo/IAO_>
PREFIX obi:     <http://purl.obolibrary.org/obo/OBI_>
PREFIX pmd:     <https://w3id.org/pmd/co/>
PREFIX qudt:    <http://qudt.org/schema/qudt/>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?model_type ?param_name ?param_value ?param_unit
WHERE {
  ?card a iao:0000100 ;
        pmd:PMD_0000104 ?model .
  ?model dcterms:type ?model_type ;
         obi:0000299 ?param .
  ?param rdfs:label ?param_name ;
         qudt:value ?param_value .
  OPTIONAL { ?param qudt:hasUnit ?param_unit . }
}
ORDER BY ?param_name
"""

rows = list(flat.query(SPARQL_MODEL))
if rows:
    model_type = str(rows[0].model_type)
    print(f"Constitutive model: {model_type}")
    print(f"  {'Parameter':<16}  {'Value':>12}  Unit")
    print("  " + "-" * 40)
    for r in rows:
        if r.param_name is None:
            continue
        val  = f"{float(r.param_value):>12.4f}"
        unit = str(r.param_unit) if r.param_unit else ""
        print(f"  {str(r.param_name):<16}  {val}  {unit}")
else:
    print("No constitutive model recorded.")

Constitutive model: Hockett-Sherby
  Parameter                Value  Unit
  ----------------------------------------
  c                      12.5000  
  n                       0.6800  
  sigma_0               220.0000  MPa
  sigma_sat             780.0000  MPa


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the material reference, constants, and model parameters |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Elastic constants, mechanical properties, and model parameters are extracted to confirm correctness |

To create a card for a different material, edit `docs/example.input.json` and re-run all cells.


## Further reading

- [Schema format reference](../../../docs/3_schema-format.md): how to write your own schema
- [Schema patterns](../../../docs/4_schema-patterns.md): inheritance and composition patterns
- `simulation/model-calibration/PMDCo/` is the step that produces this card
- `simulation/step/PMDCo/` is the FEM step that consumes this card
- See `workflow/PMDCo/docs/3_material_card_without_template.ipynb` for a cross-domain demo showing the full production-to-FEM pipeline